In [15]:
import pandas as pd
import numpy as np

w = pd.read_parquet("../data/interim/balance_sheet_wide.parquet")
w = w[w["Assets"] >= 1e6].copy()                      # décision de l'EDA 2
w = w[w["cik"] != "0000000000"].copy()                # CIK invalide
w = w.sort_values(["cik", "end"]).reset_index(drop=True)

# NOUVEAU : on ajoute les flux de la même année (même entreprise, même date de clôture)
fw = pd.read_parquet("../data/interim/flows_wide.parquet")
w = w.merge(fw, on=["cik", "end"], how="left")
print(w.shape, w["cik"].nunique())

(92831, 43) 12218


In [16]:
NOMS = {
    "Assets": "A", "Liabilities": "L", "StockholdersEquity": "E",
    "CashAndCashEquivalentsAtCarryingValue": "C",
    "AssetsCurrent": "AC", "LiabilitiesCurrent": "LC",
    "RetainedEarningsAccumulatedDeficit": "RE",
    "PropertyPlantAndEquipmentNet": "PPE",
    "Goodwill": "GW", "IntangibleAssetsNetExcludingGoodwill": "INT",
}
NOMS_FLUX = {                                                  # NOUVEAU
    "CA": "CA", "NetIncomeLoss": "NI", "OperatingIncomeLoss": "OI",
    "NetCashProvidedByUsedInOperatingActivities": "CFO",
    "NetCashProvidedByUsedInInvestingActivities": "CFI",
    "NetCashProvidedByUsedInFinancingActivities": "CFF",
    "PaymentsToAcquirePropertyPlantAndEquipment": "CAPEX",
    "InterestExpense": "INTEXP",
    "PaymentsOfDividends": "DIV",
    "PaymentsForRepurchaseOfCommonStock": "RACHATS",
}
POSITIFS = ["A", "L", "AC", "LC"]
SIGNES   = ["C", "E"]

def log_var(nouveau, ancien):
    return np.log(nouveau.where(nouveau > 0)) - np.log(ancien.where(ancien > 0))

def ratio(num, den):
    return num / den.where(den > 0)

def features_et_cibles(w):
    tous = {**NOMS, **NOMS_FLUX}                               # NOUVEAU
    b = w[["cik", "end"] + list(tous)].rename(columns=tous)
    b = b.sort_values(["cik", "end"]).reset_index(drop=True)
    cols = list(tous.values())

    g = b.groupby("cik")
    suiv = g[["end"] + cols].shift(-1)     # t+1 -> UNIQUEMENT pour les cibles
    prec = g[["end"] + cols].shift(1)      # t-1 -> pour les features
    ok_suiv = (suiv["end"] - b["end"]).dt.days.between(330, 400)
    ok_prec = (b["end"] - prec["end"]).dt.days.between(330, 400)

    out = b[["cik", "end"]].copy()

    # --- FEATURES DU BILAN (année t) ---
    out["taille"]   = np.log10(b["A"])
    out["L_A"]      = ratio(b["L"], b["A"])
    out["E_A"]      = ratio(b["E"], b["A"])
    out["C_A"]      = ratio(b["C"], b["A"])
    out["RE_A"]     = ratio(b["RE"], b["A"])
    out["AC_LC"]    = ratio(b["AC"], b["LC"])
    out["AC_A"]     = ratio(b["AC"], b["A"])
    out["LC_A"]     = ratio(b["LC"], b["A"])
    out["PPE_A"]    = ratio(b["PPE"], b["A"])
    out["incorp_A"] = ratio(b["GW"].fillna(0) + b["INT"].fillna(0), b["A"])
    out["E_negatif"] = (b["E"] < 0).astype(float).where(b["E"].notna())

    # --- NOUVEAU : FEATURES DES FLUX (année t, qui se termine à la date du bilan t) ---
    out["ROA"]           = ratio(b["NI"], b["A"])              # rentabilité des actifs
    out["perte"]         = (b["NI"] < 0).astype(float).where(b["NI"].notna())
    out["marge_op"]      = ratio(b["OI"], b["CA"])             # marge d'exploitation
    out["rotation"]      = ratio(b["CA"], b["A"])              # chiffre d'affaires / actifs
    out["CFO_A"]         = ratio(b["CFO"], b["A"])             # cash généré par l'activité
    out["CFI_A"]         = ratio(b["CFI"], b["A"])             # cash investi
    out["CFF_A"]         = ratio(b["CFF"], b["A"])             # cash de financement
    out["capex_A"]       = ratio(b["CAPEX"], b["A"])           # investissements matériels
    out["distrib_A"]     = ratio((b["DIV"].fillna(0) + b["RACHATS"].fillna(0))
                                 .where(b["CFF"].notna()), b["A"])  # dividendes + rachats
    out["cout_dette"]    = ratio(b["INTEXP"], b["L"])          # intérêts / dettes
    out["couv_interets"] = ratio(b["OI"], b["INTEXP"])         # résultat d'exploitation / intérêts
    out["accruals_A"]    = ratio(b["NI"] - b["CFO"], b["A"])   # bénéfice non transformé en cash
    out["d_CA"]          = log_var(b["CA"], prec["CA"]).where(ok_prec)   # croissance du CA

    # --- VARIATIONS PASSÉES (t-1 -> t) ---
    for k in POSITIFS:
        out[f"d_{k}"] = log_var(b[k], prec[k]).where(ok_prec)
    for k in SIGNES:
        out[f"d_{k}"] = ((b[k] - prec[k]) / prec["A"].where(prec["A"] > 0)).where(ok_prec)

    # --- CIBLES (t -> t+1) ---
    for k in POSITIFS:
        out[f"y_{k}"] = log_var(suiv[k], b[k]).where(ok_suiv)
    for k in SIGNES:
        out[f"y_{k}"] = ((suiv[k] - b[k]) / b["A"]).where(ok_suiv)

    return out

table = features_et_cibles(w)
print(table.shape)

(92831, 38)


In [17]:
cibles = [c for c in table if c.startswith("y_")]
print(table[cibles].notna().mean().round(3))          # part des lignes avec une cible
table[cibles].describe(percentiles=[.01, .05, .5, .95, .99]).round(3)

y_A     0.856
y_L     0.836
y_AC    0.655
y_LC    0.652
y_C     0.760
y_E     0.778
dtype: float64


,y_A,y_L,y_AC,y_LC,y_C,y_E
count,79480.000,77568.000,60772.000,60523.000,70566.000,72228.000
mean,0.076,0.113,0.080,0.136,0.115,0.345
std,0.475,0.619,0.738,0.654,3.843,21.988
min,-7.012,-14.035,-11.040,-7.363,-25.288,-45.140
1%,-1.182,-1.510,-1.727,-1.538,-0.531,-0.975
5%,-0.453,-0.510,-0.760,-0.625,-0.218,-0.384
50%,0.036,0.046,0.037,0.077,0.001,0.010
95%,0.774,0.978,1.041,1.076,0.381,0.899
99%,1.740,2.319,2.692,2.472,2.032,4.785
max,9.224,13.630,10.506,9.220,680.858,5599.692


In [18]:
coupure = pd.Timestamp("2018-12-31")
complet = features_et_cibles(w)
tronque = features_et_cibles(w[w["end"] <= coupure])   # on "efface le futur"

features = [c for c in complet if c not in ("cik", "end") and not c.startswith("y_")]

a = complet[complet["end"] <= coupure].set_index(["cik", "end"]).sort_index()
t = tronque.set_index(["cik", "end"]).sort_index()

# 1) Les features doivent être IDENTIQUES
pd.testing.assert_frame_equal(a[features], t[features])
print(f" Les {len(features)} features n'utilisent aucune donnée future")

# 2) Les cibles DOIVENT changer (sinon la cible ne regarderait pas t+1)
diff = (a["y_A"].notna() != t["y_A"].notna()).sum()
print(f"Cibles y_A qui disparaissent quand on efface le futur : {diff:,}  (attendu > 0)")

 Les 30 features n'utilisent aucune donnée future
Cibles y_A qui disparaissent quand on efface le futur : 4,749  (attendu > 0)


In [19]:
ok = table.dropna(subset=["y_A"])
corr = ok[features + cibles].corr(method="spearman").loc[features, cibles]
corr.round(2).style.background_gradient(cmap="RdBu", vmin=-0.5, vmax=0.5)

,y_A,y_L,y_AC,y_LC,y_C,y_E
taille,0.030000,-0.040000,0.000000,-0.070000,-0.010000,0.030000
L_A,0.030000,-0.150000,0.060000,-0.140000,0.050000,0.000000
E_A,-0.020000,0.110000,-0.050000,0.050000,-0.080000,-0.020000
C_A,0.040000,0.040000,-0.060000,0.040000,-0.160000,0.030000
RE_A,0.070000,-0.010000,0.030000,-0.040000,0.010000,0.110000
AC_LC,-0.010000,0.090000,-0.120000,0.160000,-0.110000,-0.010000
AC_A,0.060000,0.050000,-0.050000,-0.000000,-0.050000,0.030000
LC_A,0.080000,-0.090000,0.080000,-0.210000,0.080000,0.050000
PPE_A,-0.010000,-0.040000,0.010000,-0.060000,0.010000,0.030000
incorp_A,0.010000,-0.040000,0.070000,-0.050000,0.030000,0.030000


In [20]:
print(corr.round(2))

                y_A   y_L  y_AC  y_LC   y_C   y_E
taille         0.03 -0.04  0.00 -0.07 -0.01  0.03
L_A            0.03 -0.15  0.06 -0.14  0.05  0.00
E_A           -0.02  0.11 -0.05  0.05 -0.08 -0.02
C_A            0.04  0.04 -0.06  0.04 -0.16  0.03
RE_A           0.07 -0.01  0.03 -0.04  0.01  0.11
AC_LC         -0.01  0.09 -0.12  0.16 -0.11 -0.01
AC_A           0.06  0.05 -0.05 -0.00 -0.05  0.03
LC_A           0.08 -0.09  0.08 -0.21  0.08  0.05
PPE_A         -0.01 -0.04  0.01 -0.06  0.01  0.03
incorp_A       0.01 -0.04  0.07 -0.05  0.03  0.03
E_negatif      0.01 -0.04  0.04  0.03  0.07  0.04
ROA            0.13  0.00  0.06 -0.05  0.03  0.19
perte         -0.15 -0.02 -0.07  0.04 -0.03 -0.17
marge_op       0.15  0.00  0.08 -0.04  0.03  0.18
rotation       0.06 -0.03  0.06 -0.06  0.06  0.10
CFO_A          0.09 -0.03  0.07 -0.05  0.02  0.16
CFI_A         -0.18 -0.14 -0.06 -0.08 -0.06 -0.11
CFF_A          0.11  0.14  0.00  0.12 -0.02  0.01
capex_A        0.06  0.02  0.04 -0.02  0.03  0.07


In [21]:
table.to_parquet("../data/processed/model_table.parquet", index=False)
print(table.shape)

(92831, 38)


## Synthèse
- 30 features (17 du bilan + 13 des flux : rentabilité, cash, coût de la dette) (ratios à t, taille, variations t-1 → t), 6 cibles (t → t+1).
- Test anti-fuite réussi : features identiques après suppression du futur.
- Corrélations (Spearman) faibles (|ρ| ≤ 0,24) : élan (d_A → y_A = 0,24, d_E → y_E = 0,21)
  et retour à la moyenne (LC_A → y_LC = −0,21, C_A → y_C = −0,16).
- Cibles y_C et y_E très extrêmes (max 680 et 5 600) → winsorizing sur le train.
- 5 bilans au CIK invalide exclus (bug d'ingestion corrigé à la source).